# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1
import pandas as pd
import plotly.express as px

# Load gapminder data
gm = px.data.gapminder()
gm_2007 = gm[gm['year'] == 2007].copy()

# Compute deviation from global mean
global_mean = gm_2007['lifeExp'].mean()
gm_2007['lifeExp_vs_avg'] = gm_2007['lifeExp'] - global_mean

# Base choropleth
fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp_vs_avg',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    hover_name='country',
    hover_data={
        'lifeExp': ':.1f',
        'lifeExp_vs_avg': ':.1f',
        'iso_alpha': False
    },
    labels={
        'lifeExp': 'Life Expectancy (years)',
        'lifeExp_vs_avg': f'vs World Avg ({global_mean:.1f} yrs)'
    },
    title='Sub-Saharan Africa falls furthest below the world average'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='#CCCCCC',
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title='Deviation from<br>Global Average',
        ticksuffix=' yrs',
        thickness=15,
        len=0.6
    ),
    margin=dict(l=0, r=0, t=55, b=0)
)
fig.show()

# Insight print
lowest = gm_2007.sort_values('lifeExp_vs_avg').iloc[0]
print(f"Insight: {lowest['country']} has the lowest life expectancy "
      f"relative to the global average ({lowest['lifeExp_vs_avg']:.1f} years below).")


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding


### Task 2 — Design decisions

**GeoJSON source:** Local `4_niedrig.geo.json` (Federal States of Germany boundary GeoJSON).

**Chart type chosen** (`px.choropleth` or `px.choropleth_map`) **and reason:**

`px.choropleth` — Since we are presenting a regional map of Germany for a static analysis (unemployment, cost indexing, etc.), a projection-based choropleth without requiring web map tiles is cleaner, loads faster, and avoids background clutter.

**Colour scale chosen** (sequential or diverging) **and reason:**

Sequential (`Reds`) — The numeric metric represents magnitude/level (cost or rate) with no natural midpoint or zero crossing. Sequential scale (darker = higher value) is the correct choice here.


In [ ]:
# Task 2
import json
import pandas as pd
import plotly.express as px

# Load Germany States GeoJSON
with open(r'4_niedrig.geo.json', 'r', encoding='utf-8') as file:
    germany_geojson = json.load(file)

# Create dataset for German states where Bayern (DE-BY) has the lowest value (12.5)
data = [
    {'id': 'DE-BW', 'name': 'Baden-Württemberg', 'value': 20.9},
    {'id': 'DE-BY', 'name': 'Bayern', 'value': 12.5},
    {'id': 'DE-BE', 'name': 'Berlin', 'value': 13.8},
    {'id': 'DE-BB', 'name': 'Brandenburg', 'value': 16.7},
    {'id': 'DE-HB', 'name': 'Bremen', 'value': 16.1},
    {'id': 'DE-HH', 'name': 'Hamburg', 'value': 22.0},
    {'id': 'DE-HE', 'name': 'Hessen', 'value': 21.3},
    {'id': 'DE-MV', 'name': 'Mecklenburg-Vorpommern', 'value': 23.8},
    {'id': 'DE-NI', 'name': 'Niedersachsen', 'value': 14.5},
    {'id': 'DE-NW', 'name': 'Nordrhein-Westfalen', 'value': 18.4},
    {'id': 'DE-RP', 'name': 'Rheinland-Pfalz', 'value': 13.8},
    {'id': 'DE-SL', 'name': 'Saarland', 'value': 16.0},
    {'id': 'DE-ST', 'name': 'Sachsen-Anhalt', 'value': 19.3},
    {'id': 'DE-SN', 'name': 'Sachsen', 'value': 13.8},
    {'id': 'DE-SH', 'name': 'Schleswig-Holstein', 'value': 15.8},
    {'id': 'DE-TH', 'name': 'Thüringen', 'value': 21.0}
]
df_germany = pd.DataFrame(data)

# Build the choropleth map
fig = px.choropleth(
    df_germany,
    geojson=germany_geojson,
    locations='id',
    featureidkey='properties.id',
    color='value',
    color_continuous_scale='Reds',
    hover_name='name',
    hover_data={'value': ':.1f', 'id': False},
    labels={'value': 'Metric Value'},
    title='Bayern has the lowest value among all German states'
)

# Styling
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0)
)
fig.show()

# Print the insight
lowest = df_germany.sort_values('value').iloc[0]
print(f"Insight: {lowest['name']} has the lowest value ({lowest['value']:.1f}).")
